# Phase 6 runner (thin) — disease-consistency experiment (A vs B)

Runs **existing repository scripts** on Kaggle (2x T4). Rules, design and pre-registered decision rules: `docs/PHASE_6_DISEASE_CONSISTENCY.md`.

Attach as inputs: (1) the repo dataset (`scripts/package_for_kaggle.py`), (2) an authorized CheXpert-small dataset (`train.csv` + images),
(3) a dataset containing the Phase 5b classifier `best.pt` (SHA-256 `d14a9e87...763a`; the code refuses any other file). Accelerator GPU T4 x2, Internet On.

Sessions: **1** = `RUN_SMOKE` + `RUN_VIABILITY` (defaults). **2** = `RUN_TRAIN` (+ `RUN_EVAL`) after I have set `MAX_STEPS` and `LAMBDA1` from the smoke run by the pre-registered rules.
Never edit anything but the parameter cell.

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile

# ---------------- parameters (edit only this cell) ----------------
RUN_SMOKE = True        # 20-step A-ft and B smoke runs (memory, s/step, loss scale, save + resume) -> inputs of the lambda1 / steps rules
RUN_VIABILITY = True    # untouched PRISM checkpoint (A0) on 8 test images at self_replace_steps 0.2 -> "meaningful edit" rule M1-M4
RUN_TRAIN = False       # the lean A-ft and B runs (parallel, one per GPU): needs MAX_STEPS and LAMBDA1
RUN_EVAL = False        # A-ft, B (and A0) on the 24 fixed test images, then scoring + the pre-registered comparison
EVAL_A0 = True          # also evaluate the untouched checkpoint on all 24 images
MAX_STEPS = None        # optimizer steps for A-ft and B (identical)       -- set after the smoke run by the rule in the doc
LAMBDA1 = None          # weight of L_disease                              -- set after the smoke run by the rule in the doc
MICRO_BATCH, GRAD_ACCUM = 2, 4   # identical for every arm (changing micro-batch changes the noise draws, so never for one arm only)
PINNED_CLASSIFIER_SHA256 = "d14a9e87519e80803449cdeb2cf550a08b4508f52587d3767aa967f8cb51763a"
PRISM_CKPT = "/kaggle/temp/prism/model.safetensors"
RESTORE_FROM = sorted(p for d in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * d + "experiments"))   # previous output, if attached
# ------------------------------------------------------------------

WORK, REPO = "/kaggle/working", "/kaggle/working/repo"
OUT = f"{WORK}/experiments"
P5, P6 = f"{OUT}/phase5", f"{OUT}/phase6"
os.makedirs(P5, exist_ok=True); os.makedirs(P6, exist_ok=True)
os.environ.update(HF_HOME="/kaggle/temp/hf", PYTHONUNBUFFERED="1")
FAILED, LOG_PATH = [], f"{OUT}/notebook_log.txt"
LOG = open(LOG_PATH, "a")

def say(text=""):
    print(text, flush=True); LOG.write(text + "\n"); LOG.flush()

def run(name, cmd, gpu=None):
    say(); say(f"=== {name}"); say("$ " + " ".join(cmd))
    env = dict(os.environ, **({"CUDA_VISIBLE_DEVICES": str(gpu)} if gpu is not None else {}))
    proc = subprocess.Popen(cmd, cwd=REPO if os.path.isdir(REPO) else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        say(line.rstrip("\n"))
    rc = proc.wait(); say(f"--- exit code {rc}")
    if rc: FAILED.append((name, rc))
    return rc

def run_par(jobs):
    # jobs: [(name, cmd, gpu)] run concurrently, one per GPU (sequentially if fewer GPUs); output prefixed with [name]
    import torch
    if torch.cuda.device_count() < len(jobs):
        say(f"only {torch.cuda.device_count()} GPU(s): running {len(jobs)} jobs one after another")
        return [run(n, c, 0) for n, c, _ in jobs]
    procs = []
    for name, cmd, gpu in jobs:
        say(); say(f"=== {name} (GPU {gpu})"); say("$ " + " ".join(cmd))
        procs.append((name, subprocess.Popen(cmd, cwd=REPO, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)))
    def pump(name, p):
        for line in p.stdout: say(f"[{name}] " + line.rstrip("\n"))
    threads = [threading.Thread(target=pump, args=x) for x in procs]
    [t.start() for t in threads]; [t.join() for t in threads]
    rcs = [p.wait() for _, p in procs]
    for (name, _), rc in zip(procs, rcs):
        say(f"--- [{name}] exit code {rc}")
        if rc: FAILED.append((name, rc))
    return rcs

In [ ]:
def find(suffix):
    return sorted((p for d in range(1, 5) for p in glob.glob("/kaggle/input/" + "*/" * d + suffix)), key=len)

say(f"notebook flags: SMOKE={RUN_SMOKE} VIABILITY={RUN_VIABILITY} TRAIN={RUN_TRAIN} EVAL={RUN_EVAL} MAX_STEPS={MAX_STEPS} LAMBDA1={LAMBDA1} micro_batch={MICRO_BATCH} accum={GRAD_ACCUM}")
boots = find("scripts/kaggle_bootstrap.py")
if not boots:
    zips = find("*repo*.zip")
    if zips:
        shutil.unpack_archive(zips[0], "/kaggle/temp/repo_src"); boots = ["/kaggle/temp/repo_src/scripts/kaggle_bootstrap.py"]
assert boots, "repo package not found under /kaggle/input: attach the dataset built by scripts/package_for_kaggle.py"
say(f"using repo package: {boots[0]}")
run("bootstrap (copy repo, install deps, torch guarded)", [sys.executable, boots[0], "--repo-src", os.path.dirname(os.path.dirname(boots[0])), "--workdir", REPO])
run("environment check", [sys.executable, "scripts/check_environment.py", "--require-gpu", "--min-vram-gb", "8", "--out", f"{OUT}/env_report.json"])

In [ ]:
READY = False
clf = [p for p in find("best.pt") if hashlib.sha256(open(p, "rb").read()).hexdigest() == PINNED_CLASSIFIER_SHA256]
assert clf, "no best.pt with SHA-256 " + PINNED_CLASSIFIER_SHA256 + " under /kaggle/input: attach the Phase 5b classifier checkpoint"
CLF = clf[0]; say(f"frozen classifier: {CLF} (hash verified)")
if not os.path.exists(PRISM_CKPT):
    run("fetch PRISM checkpoint (3.4 GB, size + SHA-256 verified)", [sys.executable, "scripts/download_prism_checkpoint.py", "--dest", os.path.dirname(PRISM_CKPT)])
LOCATED = run("locate CheXpert + write Phase 5 configs", [sys.executable, "scripts/locate_chexpert.py", "--out-dir", P5, "--prism-checkpoint", PRISM_CKPT]) == 0
if LOCATED:
    run("Phase 6 configs + fixed evaluation set", [sys.executable, "scripts/make_phase6_inputs.py", "--phase5-dir", P5, "--prism-checkpoint", PRISM_CKPT, "--classifier-checkpoint", CLF, "--out-dir", P6])
READY = os.path.exists(f"{P6}/generated_phase6_B.yaml")       # the fine-tune configs are written first: an eval-set problem cannot block the smoke run
EVAL_READY = os.path.exists(f"{P6}/eval_set.json")
SETS = ["--set", f"finetune.micro_batch={MICRO_BATCH}", "--set", f"finetune.grad_accum={GRAD_ACCUM}"]
def ft_cmd(arm, name, extra):
    return [sys.executable, "scripts/run_experiment.py", "--config", f"{P6}/generated_phase6_{arm}.yaml", "--name", name, "--run-root", OUT, *SETS, *extra]

In [ ]:
if RUN_SMOKE and READY:
    sm = ["--set", "finetune.max_steps=20", "--set", "finetune.save_every=10", "--set", "finetune.log_every=2"]
    run_par([("smoke A-ft", ft_cmd("A", "phase6_smoke_A", sm), 0),
             ("smoke B (weight 0.3 is a smoke-only placeholder)", ft_cmd("B", "phase6_smoke_B", sm + ["--set", "losses.disease_consistency.weight=0.3"]), 1)])
    # save + resume on GPU: continue the B smoke run from its step-20 checkpoint to step 24
    run("smoke B resume (20 -> 24 steps)", ft_cmd("B", "phase6_smoke_B", ["--set", "finetune.max_steps=24", "--set", "finetune.save_every=10", "--set", "finetune.log_every=2",
                                                                         "--set", "losses.disease_consistency.weight=0.3", "--restore-from", OUT]), 0)
    run("smoke PASS/FAIL against the pre-registered smoke criteria", [sys.executable, "scripts/check_phase6_smoke.py", "--run-root", OUT, "--out", f"{P6}/smoke_check.json"])

In [ ]:
def eval_cmd(config, name, cache, delta=None):
    cmd = [sys.executable, "scripts/run_experiment.py", "--config", f"{P6}/{config}", "--name", name, "--run-root", OUT, "--set", f"editing.inversion_cache_dir={OUT}/phase6/inversion_cache/{cache}"]
    return cmd + (["--set", f"model.delta_checkpoint_path={delta}"] if delta else [])

def score(arm, run_name):
    d = sorted(glob.glob(f"{OUT}/{run_name}/*"))[-1]
    return run(f"score {arm}", [sys.executable, "scripts/evaluate_counterfactuals.py", "--run-dir", d, "--eval-set", f"{P6}/eval_set.json", "--classifier", CLF,
                                "--classifier-sha", PINNED_CLASSIFIER_SHA256, "--arm", arm, "--out-dir", f"{P6}/scores"])

if RUN_VIABILITY and EVAL_READY:
    if run("A0 viability: untouched PRISM checkpoint, 8 test images, self_replace_steps 0.2", eval_cmd("generated_eval_viability.yaml", "phase6_viability_A0", "A0")) == 0 and score("A0_viability", "phase6_viability_A0") == 0:
        run("pre-registered 'meaningful edit' rule on A0", [sys.executable, "scripts/compare_arms.py", "--A0", f"{P6}/scores/per_image_A0_viability.csv", "--out", f"{P6}/scores/compare_viability.json"])

In [ ]:
if RUN_TRAIN and READY:
    assert MAX_STEPS and LAMBDA1 is not None, "set MAX_STEPS and LAMBDA1 (rules in docs/PHASE_6_DISEASE_CONSISTENCY.md) before RUN_TRAIN"
    tr = ["--set", f"finetune.max_steps={MAX_STEPS}"] + (["--restore-from", *RESTORE_FROM] if RESTORE_FROM else [])
    run_par([("train A-ft", ft_cmd("A", "phase6_A_ft", tr), 0),
             ("train B", ft_cmd("B", "phase6_B", tr + ["--set", f"losses.disease_consistency.weight={LAMBDA1}"]), 1)])

In [ ]:
def delta_of(name):
    c = sorted((p for root in [OUT, *RESTORE_FROM] for p in glob.glob(f"{root}/{name}/*/checkpoints/delta_final.safetensors")), key=lambda p: p.split("/")[-3])
    assert c, f"no trained delta for {name}: run RUN_TRAIN first"
    return c[-1]

if RUN_EVAL and EVAL_READY:
    jobs = [("eval A-ft", eval_cmd("generated_eval.yaml", "phase6_eval_A_ft", "A_ft", delta_of("phase6_A_ft")), 0),
            ("eval B", eval_cmd("generated_eval.yaml", "phase6_eval_B", "B", delta_of("phase6_B")), 1)]
    run_par(jobs)
    if EVAL_A0:
        run("eval A0 (untouched checkpoint), all 24 images", eval_cmd("generated_eval.yaml", "phase6_eval_A0", "A0"))
    ok = [score(a, n) == 0 for a, n in (("A_ft", "phase6_eval_A_ft"), ("B", "phase6_eval_B"))] + ([score("A0", "phase6_eval_A0") == 0] if EVAL_A0 else [])
    if all(ok):
        cmp = [sys.executable, "scripts/compare_arms.py", "--A-ft", f"{P6}/scores/per_image_A_ft.csv", "--B", f"{P6}/scores/per_image_B.csv", "--out", f"{P6}/scores/compare.json"]
        run("pre-registered comparison", cmp + (["--A0", f"{P6}/scores/per_image_A0.csv"] if EVAL_A0 else []))

In [ ]:
# Always archive (even after a failure). Checkpoints > 50 MB stay in /kaggle/working (indexed with SHA-256 in each run's artifacts.json) so the zip stays downloadable.
stamp = time.strftime('%Y%m%d-%H%M%S'); archive = f"{WORK}/artifacts_{stamp}.zip"; skipped = []
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            p = os.path.join(root, f)
            if f.endswith((".pt", ".safetensors")) and os.path.getsize(p) > 50e6:
                skipped.append(os.path.relpath(p, OUT)); continue
            if os.sep + "inversion_cache" + os.sep in p or f == "train_resolved.csv":
                continue
            z.write(p, os.path.relpath(p, OUT))
say(f"artifacts: {archive} {os.path.getsize(archive) / 1e6:.2f} MB; large checkpoints left out of the zip: {skipped}")
say(f"artifacts sha256: {hashlib.file_digest(open(archive, 'rb'), 'sha256').hexdigest()}   (verify the downloaded zip against this line)")
say(f"failed steps: {FAILED or 'none'}")
if FAILED:
    raise RuntimeError(f"failed steps: {FAILED}")